# 性能分析
分析代码和资源之间的关系

**两种方法论**
- event-based profiling：像探针一样插入代码中，计算时间（如cProfile）
- statistical profiling：分析日志（如flamegraph）

**几个概念**
* CPU: ring0（核心）, ring1（虚拟层）, ring2（虚拟层）, ring3（用户）
* CPU 核心凭证: 用户凭此凭证去访问核心，进入方式有中断门, 调用门, 陷阱门和任务门

## Event-based profiling

### `sys.setprofile(profiler)`

In [1]:
import profile
import sys

def profiler(frame, event, arg):
    print(f'PROFILER: {event} {arg}')
    pass

sys.setprofile(profiler)

def fib(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fib(n - 1) + fib(n - 2)

def fib_seq(n):
    seq = []
    if n > 0:
        seq.extend(fib_seq(n - 1))
    seq.append(fib(n))
    return seq

fib_seq(2)


PROFILER: return None
PROFILER: return False
PROFILER: call None
PROFILER: call None
PROFILER: call None
PROFILER: return True
PROFILER: return True
PROFILER: call None
PROFILER: return True
PROFILER: return True
PROFILER: c_call <built-in function getattr>
PROFILER: c_return <built-in function getattr>
PROFILER: call None
PROFILER: call None
PROFILER: c_call <built-in function getattr>
PROFILER: c_return <built-in function getattr>
PROFILER: return None
PROFILER: return <contextlib._GeneratorContextManager object at 0x7f2a203a1b70>
PROFILER: call None
PROFILER: c_call <built-in function next>
PROFILER: call None
PROFILER: return None
PROFILER: c_return <built-in function next>
PROFILER: return None
PROFILER: call None
PROFILER: c_call <built-in function compile>
PROFILER: c_return <built-in function compile>
PROFILER: return <code object <module> at 0x7f2a229f19a0, file "/tmp/ipykernel_42223/2347210060.py", line 1>
PROFILER: call None
PROFILER: return False
PROFILER: call None
PROFILE

[0, 1, 1]

PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a204468c0>
PROFILER: call None
PROFILER: c_call <built-in method acquire of _thread.lock object at 0x7f2a20446900>
PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a20446900>
PROFILER: return None
PROFILER: return True
PROFILER: call None
PROFILER: c_call <built-in method __exit__ of _thread.lock object at 0x7f2a20446900>
PROFILER: c_return <built-in method __exit__ of _thread.lock object at 0x7f2a20446900>
PROFILER: return None
PROFILER: return True
PROFILER: return None
PROFILER: call None
PROFILER: call None
PROFILER: call None
PROFILER: return True
PROFILER: call None
PROFILER: c_call <built-in method acquire of _thread.lock object at 0x7f2a21dc0680>
PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a21dc0680>
PROFILER: return None
PROFILER: return True
PROFILER: call None
PROFILER: return 139818835113728
PROFILER: call None
PROFILER: c_call <built-in functi

PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a203be5c0>
PROFILER: call None
PROFILER: c_call <built-in method acquire of _thread.lock object at 0x7f2a203be7c0>
PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a203be7c0>
PROFILER: return None
PROFILER: return True
PROFILER: call None
PROFILER: c_call <built-in method __exit__ of _thread.lock object at 0x7f2a203be7c0>
PROFILER: c_return <built-in method __exit__ of _thread.lock object at 0x7f2a203be7c0>
PROFILER: return None
PROFILER: return True
PROFILER: return None
PROFILER: call None
PROFILER: call None
PROFILER: call None
PROFILER: return True
PROFILER: call None
PROFILER: c_call <built-in method acquire of _thread.lock object at 0x7f2a21dc0680>
PROFILER: c_return <built-in method acquire of _thread.lock object at 0x7f2a21dc0680>
PROFILER: return None
PROFILER: return True
PROFILER: call None
PROFILER: return 139818835113728
PROFILER: call None
PROFILER: c_call <built-in functi

### `cProfile`

In [7]:
import cProfile

def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

def fib_seq(n):
    seq = []
    if n > 0:
        seq.extend(fib_seq(n - 1))
    seq.append(fibonacci(n))
    return seq

cProfile.run('fib_seq(30)')

         7049218 function calls (96 primitive calls) in 1.840 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     31/1    0.000    0.000    1.840    1.840 1635120769.py:11(fib_seq)
7049123/31    1.840    0.000    1.840    0.059 1635120769.py:3(fibonacci)
        1    0.000    0.000    1.840    1.840 <string>:1(<module>)
        1    0.000    0.000    1.840    1.840 {built-in method builtins.exec}
       31    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
       30    0.000    0.000    0.000    0.000 {method 'extend' of 'list' objects}




In [11]:
import cProfile

def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

def fib_seq(n):
    seq = []
    if n > 0:
        seq.extend(fib_seq(n - 1))
    seq.append(fibonacci(n))
    return seq

def my_func():
    cProfile.runctx('fib_seq(30)', globals(), locals())

prof = cProfile.Profile()
prof.enable()

my_func()
prof.create_stats()
prof.print_stats()


         3 function calls in 0.000 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.000    0.000 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}




NameError: name 'fib_seq' is not defined